# Lesson 03 — Why AI Works Now

## What this notebook does

This notebook builds two small, from-scratch demos that make the "three ingredients" idea concrete instead of abstract. Part 1 trains the same word-counting urgency classifier from `ticket_urgency.py` on a tiny training set versus the full training set, and measures accuracy on the same held-out test tickets — a hands-on look at why more **data** helps a model learn a real pattern instead of noise. Part 2 counts how many basic word-lookup operations training would need at a few different dataset sizes, to build a felt sense of why **computing power** (specifically, chips that can do huge numbers of these calculations at once) became the second ingredient AI needed. No external libraries are used — only Python's standard library (`collections.Counter`).

## Part 1 — The data ingredient: does more data actually help?

We reuse the exact same approach as `customer-support-assistant/ticket_urgency.py`: count which words show up in `urgent` versus `not urgent` training tickets, then score a new ticket by adding up how urgent-leaning or not-urgent-leaning its words were during training. The only thing we change between the two runs below is **how much training data the model gets to learn from** — everything else about the method stays identical. That isolates data as the one variable, so any difference in accuracy is data's doing, not a smarter algorithm.

In [ ]:
from collections import Counter

FULL_TRAINING_TICKETS = [
    ("My order still has not arrived and I need it right now", "urgent"),
    ("This is extremely time sensitive please help right now", "urgent"),
    ("The app crashed and I cannot log in please fix this immediately", "urgent"),
    ("My package was supposed to arrive yesterday and now it says lost", "urgent"),
    ("Just wondering when my package will ship no rush", "not urgent"),
    ("Can you tell me more about your return policy sometime", "not urgent"),
    ("I have a general question about sizing whenever you get a chance", "not urgent"),
    ("Do you offer gift wrapping for orders", "not urgent"),
]

TINY_TRAINING_TICKETS = [FULL_TRAINING_TICKETS[0], FULL_TRAINING_TICKETS[4]]


def tokenize(text):
    return text.lower().replace(",", "").replace(".", "").split()


def build_word_counts(training_tickets):
    urgent_counts = Counter()
    not_urgent_counts = Counter()
    for text, label in training_tickets:
        words = tokenize(text)
        if label == "urgent":
            urgent_counts.update(words)
        else:
            not_urgent_counts.update(words)
    return urgent_counts, not_urgent_counts


def urgency_score(text, urgent_counts, not_urgent_counts):
    words = tokenize(text)
    score = 0
    for w in words:                                    # add urgent-leaning, subtract calm-leaning
        score += urgent_counts.get(w, 0)
        score -= not_urgent_counts.get(w, 0)
    return score

# Compact alternative (same result) - the loop above as one sum():
#   return sum(urgent_counts.get(w, 0) - not_urgent_counts.get(w, 0) for w in words)


def predict_urgency(text, urgent_counts, not_urgent_counts):
    score = urgency_score(text, urgent_counts, not_urgent_counts)
    if score > 0:                                      # positive score leans urgent
        return "urgent"
    else:
        return "not urgent"

# Compact alternative (same result):
#   return "urgent" if score > 0 else "not urgent"

Now we test both versions — one trained on just 2 tickets, one trained on all 8 — against the same 4 brand-new test tickets the model has never seen, and check each prediction against the true label.

In [ ]:
TEST_TICKETS = [
    ("The app crashed please help me immediately", "urgent"),
    ("Whenever you get a chance, do you offer gift wrapping", "not urgent"),
    ("My order still has not arrived and I need help right now", "urgent"),
    ("Just wondering about your summer sale, no rush", "not urgent"),
]


def evaluate(training_tickets, test_tickets):
    urgent_counts, not_urgent_counts = build_word_counts(training_tickets)
    correct = 0
    for text, true_label in test_tickets:
        predicted = predict_urgency(text, urgent_counts, not_urgent_counts)
        if predicted == true_label:                    # did we get this one right?
            correct += 1                               # count it
            mark = "right"
        else:
            mark = "WRONG"
        print(f"  {mark}  predicted={predicted:<11} true={true_label:<11} {text}")
    print(f"  accuracy: {correct}/{len(test_tickets)}")

# Compact alternative for the if/else above (same result):
#   is_correct = predicted == true_label
#   correct += is_correct                 # True counts as 1, False as 0
#   mark = "right" if is_correct else "WRONG"


print(f"Trained on {len(TINY_TRAINING_TICKETS)} tickets:")
evaluate(TINY_TRAINING_TICKETS, TEST_TICKETS)

print()
print(f"Trained on {len(FULL_TRAINING_TICKETS)} tickets:")
evaluate(FULL_TRAINING_TICKETS, TEST_TICKETS)

## Part 2 — The compute ingredient: why these calculations need a fast machine

Even our tiny word-counting model does one lookup per word per ticket. That is a trivial amount of work for 8 tickets — but real training sets are not 8 tickets, they are thousands, millions, or more. This cell counts how many word lookups the *same simple method* would need to just score every ticket once, at a few different dataset sizes, using the average ticket length we already have.

In [ ]:
# Average words per ticket, computed with a plain loop.
total_words = 0
for text, label in FULL_TRAINING_TICKETS:              # every ticket
    total_words += len(tokenize(text))                 # add its word count
average_words_per_ticket = total_words / len(FULL_TRAINING_TICKETS)

# Compact alternative (same result):
#   average_words_per_ticket = sum(len(tokenize(text)) for text, _ in FULL_TRAINING_TICKETS) / len(FULL_TRAINING_TICKETS)

print(f"average words per ticket: {average_words_per_ticket}")
print()

dataset_sizes = [8, 1_000, 1_000_000]
for size in dataset_sizes:                             # try a few dataset sizes
    total_word_lookups = round(average_words_per_ticket * size)
    print(f"{size:>9,} tickets -> about {total_word_lookups:>11,} word lookups just to score them once")

83 lookups is nothing — a laptop does that instantly, even by accident. But 10 million lookups, for a dataset only slightly larger than what real companies actually have, is already a different story if you're doing them one at a time in a simple loop. And this toy method is far simpler than a real neural network, which repeats numeric calculations like this over and over across millions or billions of internal numbers, for every single example, many times during training.

This is where the second ingredient comes in. A regular processor (a **CPU**, the general-purpose chip in every computer) mostly does calculations one at a time, very fast. A **GPU** (Graphics Processing Unit — a chip originally built to draw thousands of pixels on screen at once for video games) is built the opposite way: it does *huge numbers* of simple calculations **at the same time** instead of one after another. Training a model is mostly the same small calculation repeated an enormous number of times, which turns out to be exactly the shape of problem GPUs are good at. That mismatch — huge repeated arithmetic versus a chip built for exactly that — is a big part of why deep learning became practical only once GPUs were being used for it, not before. The exact speed of any specific chip changes constantly and is not something to memorize here; the durable idea is the *shape* of the advantage: many small calculations, done in parallel, instead of one at a time.